Phase 6: build the small demo bundle that the API and the Streamlit app read (data/demo/).

Demo models (all evaluated earlier on the test window, see reports/eda_summary.md 5e/5f):
  XGBoost = average probability of the 3 final boosters (test PR-AUC 0.442);  GNN = 3-seed average probability (0.281);
  Blend   = 0.4 * GNN + 0.6 * XGBoost on standardised logits, using the stored validation constants (0.505).
The notebook RECOMPUTES those scores from the stored recipe and asserts that it reproduces the reported test PR-AUC, so the
numbers shown in the app are exactly the numbers computed from the bundle.

Inputs (attach as Kaggle inputs):
  papertrail-processed   transactions.parquet
  papertrail-scores      gnn_scores.parquet, xgb_scores.parquet, blend_summary.json, xgb_summary.json, xgb_v2_final_seed0/1/2.json
                         (add the last four files to the dataset: Dataset -> New Version -> add files)
  ibm-transactions-for-anti-money-laundering-aml   HI-Small_Patterns.txt (ground-truth typologies)
CPU, Internet ON, DAGSHUB_TOKEN secret attached. About 10 minutes. Download demo_bundle.zip from the Output tab and unzip it
into PaperTrail/data/demo/ (it must stay under 50 MB; the notebook checks).



In [ ]:
!pip install -q git+https://github.com/abhibarnwal707/PaperTrail.git
import os
from kaggle_secrets import UserSecretsClient
os.environ["DAGSHUB_USER_TOKEN"] = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
import dagshub, mlflow
dagshub.init(repo_owner="abhibarnwal707", repo_name="papertrail", mlflow=True)


In [ ]:
import time as _t; print("[cell 2: config] started", _t.strftime("%H:%M:%S"))
import glob
import json
import os
import shutil

import numpy as np
import polars as pl
import xgboost as xgb
from sklearn.metrics import precision_recall_curve

from papertrail.features import build_features, fit_categories
from papertrail.graph_features import HISTORY_FEATURES, add_history_features
from papertrail.metrics import evaluate
from papertrail.split import assert_valid_split, split_summary, temporal_split

KNOWN = {   # exact locations first: a full recursive search of /kaggle/input can take over a minute
    "transactions.parquet": "/kaggle/input/datasets/abhibarnwal707/papertrail-processed/",
    "HI-Small_Patterns.txt": "/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/",
}
for _n in ("gnn_scores.parquet", "xgb_scores.parquet", "blend_summary.json", "xgb_summary.json",
           "xgb_v2_final_seed0.json", "xgb_v2_final_seed1.json", "xgb_v2_final_seed2.json"):
    KNOWN[_n] = "/kaggle/input/datasets/abhibarnwal707/papertrail-scores/"


def find(name):
    exact = KNOWN.get(name, "") + name
    if os.path.exists(exact):
        return exact
    for pattern in (f"/kaggle/input/datasets/*/*/{name}", f"/kaggle/input/*/{name}", f"/kaggle/input/**/{name}"):
        hits = glob.glob(pattern, recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"{name} not found under /kaggle/input: attach the dataset that holds it")


OUT = "/kaggle/working/demo"
os.makedirs(f"{OUT}/model", exist_ok=True)
N_SHOWCASE, MAX_TXN, HOP1_CAP, EDGE_CAP, HIST_N, SHAP_TOP = 200, 400, 100, 150, 50, 10
pl.Config.set_tbl_cols(-1)


In [ ]:
import time as _t; print("[cell 3: features] started", _t.strftime("%H:%M:%S"))
df = pl.read_parquet(find("transactions.parquet")).sort(["ts", "txn_id"])
summary = split_summary(df)
assert_valid_split(summary)
dfx = add_history_features(df)
train, val, test = temporal_split(dfx)
cats = fit_categories(train)
FEATURES_HISTORY = [c for c in HISTORY_FEATURES if not c.endswith("age_s")]
tab_cols = build_features(train.head(5), cats).columns
feature_names = tab_cols + FEATURES_HISTORY
xs = json.load(open(find("xgb_summary.json")))
assert feature_names == xs["features"], "feature order differs from the trained models"
boosters = []
for s in xs["seeds"]:
    b = xgb.Booster()
    b.load_model(find(f"xgb_v2_final_seed{s}.json"))
    boosters.append(b)
print(len(feature_names), "features,", len(boosters), "boosters")


def matrix(part):
    return pl.concat([build_features(part, cats), part.select(FEATURES_HISTORY)], how="horizontal").cast(pl.Float32).to_numpy()


In [ ]:
import time as _t; print("[cell 4: scores] started", _t.strftime("%H:%M:%S"))
bs = json.load(open(find("blend_summary.json")))
R, ENS = bs["blend_recipe"], bs["ensemble"]
gnn = pl.read_parquet(find("gnn_scores.parquet")).select("txn_id", "split", "gnn_mean")
xg = pl.read_parquet(find("xgb_scores.parquet")).select("txn_id", "xgb_mean")
sc = gnn.join(xg, on="txn_id")


def logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))


xz = (logit(sc["xgb_mean"].to_numpy()) - R["xgb_mu"]) / R["xgb_sd"]
gz = (logit(sc["gnn_mean"].to_numpy()) - R["gnn_mu"]) / R["gnn_sd"]
bz = R["w_gnn"] * gz + (1 - R["w_gnn"]) * xz
sc = sc.with_columns(pl.Series("xgb_z", xz), pl.Series("gnn_z", gz), pl.Series("blend_z", bz))
THR = {n: ENS[n]["threshold"] for n in ("xgb", "gnn", "blend")}

y_test_all = df.select("txn_id", "label").join(sc.filter(pl.col("split") == "test").select("txn_id"), on="txn_id")
test_sc = sc.filter(pl.col("split") == "test").join(y_test_all, on="txn_id")
for name, col in (("xgb", "xgb_z"), ("gnn", "gnn_z"), ("blend", "blend_z")):
    e = evaluate(test_sc["label"].to_numpy(), test_sc[col].to_numpy(), THR[name])
    assert abs(e["pr_auc"] - ENS[name]["pr_auc"]) < 1e-3, f"{name}: recomputed PR-AUC {e['pr_auc']:.4f} != reported {ENS[name]['pr_auc']:.4f}"
    print(f"{name:6s} reproduces the reported test PR-AUC: {e['pr_auc']:.4f} (reported {ENS[name]['pr_auc']:.4f})")

# percentile of every scored transaction within the test-window distribution ("riskier than X% of transactions")
for col in ("xgb_z", "gnn_z", "blend_z"):
    ref = np.sort(test_sc[col].to_numpy())
    sc = sc.with_columns(pl.Series(col.replace("_z", "_pct"), np.searchsorted(ref, sc[col].to_numpy()) / len(ref)))


In [ ]:
import time as _t; print("[cell 5: typologies] started", _t.strftime("%H:%M:%S"))
rows, typ = [], None
with open(find("HI-Small_Patterns.txt")) as fh:
    for line in fh:
        line = line.strip()
        if line.startswith("BEGIN"):
            typ = line.split(" - ")[1].split(":")[0].strip()
        elif line and not line.startswith("END"):
            f = line.split(",")
            rows.append((typ, f[0], f"{int(f[1])}_{f[2]}", f"{int(f[3])}_{f[4]}", float(f[7])))
KEY = ["ts", "src", "dst", "amt_paid"]
pat = (pl.DataFrame(rows, schema=["typology"] + KEY, orient="row").with_columns(pl.col("ts").str.strptime(pl.Datetime, "%Y/%m/%d %H:%M"))
         .unique(subset=KEY, keep="first", maintain_order=True))
BASE = ["txn_id", "ts", "src", "dst", "amt_paid", "cur_paid", "cur_recv", "pay_format", "from_bank", "to_bank", "label"]
E = (df.select(BASE).join(pat, on=KEY, how="left")
       .with_columns(pl.when(pl.col("label") == 1).then(pl.col("typology")).alias("typology"))
       .join(sc.select("txn_id", "split", pl.col("xgb_mean").alias("xgb_p"), pl.col("gnn_mean").alias("gnn_p"), "xgb_z", "gnn_z", "blend_z",
                       "xgb_pct", "gnn_pct", "blend_pct"), on="txn_id", how="left"))
E = E.with_columns(*[(pl.col(f"{m}_z") >= THR[m]).fill_null(False).alias(f"{m}_flag") for m in ("xgb", "gnn", "blend")])
print(E.height, "transactions;", int((E["split"] == "test").sum()), "scored in the test window")


In [ ]:
import time as _t; print("[cell 6: showcase] started", _t.strftime("%H:%M:%S"))
T = E.filter(pl.col("split") == "test")
ends_test = pl.concat([T.select(acct=pl.col("src"), txn_id=pl.col("txn_id"), blend_z=pl.col("blend_z"), xgb_z=pl.col("xgb_z"), gnn_z=pl.col("gnn_z"),
                                blend_flag=pl.col("blend_flag"), xgb_flag=pl.col("xgb_flag"), gnn_flag=pl.col("gnn_flag"), label=pl.col("label"), typology=pl.col("typology")),
                       T.select(acct=pl.col("dst"), txn_id=pl.col("txn_id"), blend_z=pl.col("blend_z"), xgb_z=pl.col("xgb_z"), gnn_z=pl.col("gnn_z"),
                                blend_flag=pl.col("blend_flag"), xgb_flag=pl.col("xgb_flag"), gnn_flag=pl.col("gnn_flag"), label=pl.col("label"), typology=pl.col("typology"))])
acct = ends_test.group_by("acct").agg(
    n_txn=pl.len(),
    blend_risk=pl.col("blend_z").max(), xgb_risk=pl.col("xgb_z").max(), gnn_risk=pl.col("gnn_z").max(),
    n_flag_blend=pl.col("blend_flag").sum(), n_flag_xgb=pl.col("xgb_flag").sum(), n_flag_gnn=pl.col("gnn_flag").sum(),
    n_laundering=pl.col("label").sum(), typologies=pl.col("typology").drop_nulls().unique(),
).filter(pl.col("n_txn").is_between(2, MAX_TXN))        # no giant hubs: their graphs are unreadable and their max score is trivially high
acct = acct.sort("blend_risk", descending=True)
print(acct.height, "candidate accounts (2 to", MAX_TXN, "transactions in the test window)")

chosen, tags = [], {}


def add(accounts, tag):
    for a in accounts:
        if a not in tags:
            chosen.append(a)
            tags[a] = tag


all_typologies = sorted({t for ts_ in acct["typologies"].to_list() for t in ts_})
for t in all_typologies:
    hit = acct.filter(pl.col("typologies").list.contains(t)).head(15)["acct"].to_list()
    add(hit, f"true positive: {t}")
add(acct.filter(pl.col("n_laundering") == 0).head(25)["acct"].to_list(), "false positive")
add(acct.head(N_SHOWCASE)["acct"].to_list(), "top risk")
chosen = chosen[:N_SHOWCASE] if len(chosen) > N_SHOWCASE else chosen
showcase = pl.DataFrame({"account": chosen, "tag": [tags[a] for a in chosen]})
print(len(chosen), "showcase accounts; tags:", showcase.group_by("tag").len().sort("len", descending=True).head(12).to_dicts())


In [ ]:
import time as _t; print("[cell 7: evidence] started", _t.strftime("%H:%M:%S"))
S = showcase["account"].to_list()
slim = E.select("txn_id", "src", "dst", "ts", "blend_z")
ends_all = pl.concat([slim.select(acct=pl.col("src"), other=pl.col("dst"), txn_id=pl.col("txn_id"), ts=pl.col("ts"), key=pl.col("blend_z").fill_null(-1e9)),
                      slim.select(acct=pl.col("dst"), other=pl.col("src"), txn_id=pl.col("txn_id"), ts=pl.col("ts"), key=pl.col("blend_z").fill_null(-1e9))])

hop1 = (ends_all.filter(pl.col("acct").is_in(S)).sort(["acct", "key", "ts"], descending=[False, True, True])
                .group_by("acct", maintain_order=True).head(HOP1_CAP))
pairs = hop1.select(center=pl.col("acct"), nbr=pl.col("other")).unique().filter(pl.col("nbr") != pl.col("center"))
nbrs = pairs["nbr"].unique().to_list()
cand = (ends_all.filter(pl.col("acct").is_in(nbrs)).sort(["acct", "key", "ts"], descending=[False, True, True])
                .group_by("acct", maintain_order=True).head(3).select(nbr=pl.col("acct"), txn_id=pl.col("txn_id"), key=pl.col("key")))
hop2 = pairs.join(cand, on="nbr").select(center=pl.col("center"), txn_id=pl.col("txn_id"), key=pl.col("key"), hop=pl.lit(2))
edges = (pl.concat([hop1.select(center=pl.col("acct"), txn_id=pl.col("txn_id"), key=pl.col("key"), hop=pl.lit(1)), hop2])
           .group_by(["center", "txn_id"]).agg(hop=pl.col("hop").min(), key=pl.col("key").max())
           .sort(["center", "hop", "key"], descending=[False, False, True]).group_by("center", maintain_order=True).head(EDGE_CAP)
           .select("center", "txn_id", "hop"))
print("edges:", edges.height, "rows;", edges.group_by("center").len()["len"].median(), "per account (median)")

history = (ends_all.filter(pl.col("acct").is_in(S)).sort(["acct", "ts"], descending=[False, True])
                   .group_by("acct", maintain_order=True).head(HIST_N)
                   .select(account=pl.col("acct"), txn_id=pl.col("txn_id")).with_columns(rank=pl.int_range(pl.len()).over("account") + 1))

# highest-risk test transaction per showcase account -> SHAP drivers (average contribution over the 3 boosters)
top_txn = (ends_test.filter(pl.col("acct").is_in(S)).sort(["acct", "blend_z"], descending=[False, True])
                    .group_by("acct", maintain_order=True).first().select(account=pl.col("acct"), txn_id=pl.col("txn_id")))
part = dfx.filter(pl.col("txn_id").is_in(top_txn["txn_id"].to_list()))
X = matrix(part)
d = xgb.DMatrix(X, feature_names=feature_names)
contribs = np.mean([b.predict(d, pred_contribs=True) for b in boosters], axis=0)                  # (n, features + bias)
margin = boosters[0].predict(d, output_margin=True)
assert np.allclose(boosters[0].predict(d, pred_contribs=True).sum(axis=1), margin, atol=1e-3), "contributions do not add up to the model output"
pos = {t: i for i, t in enumerate(part["txn_id"].to_list())}
shap_rows = []
for a, t in zip(top_txn["account"].to_list(), top_txn["txn_id"].to_list()):
    i = pos[t]
    for j in np.argsort(-np.abs(contribs[i, :-1]))[:SHAP_TOP]:
        v = float(X[i, j])
        shap_rows.append({"account": a, "txn_id": t, "feature": feature_names[j], "value": None if np.isnan(v) else v, "contribution": float(contribs[i, j])})
shap = pl.DataFrame(shap_rows, schema={"account": pl.String, "txn_id": pl.Int64, "feature": pl.String, "value": pl.Float64, "contribution": pl.Float64})
print("shap rows:", shap.height)


In [ ]:
import time as _t; print("[cell 8: assets] started", _t.strftime("%H:%M:%S"))
labels = {
    "log_amt": ("Payment amount (log scale)", "amount"), "hour": ("Hour of the day", "hour"),
    "is_ach": ("Paid by ACH transfer", "flag"), "is_cross_currency": ("Currency differs between sender and receiver", "flag"),
    "is_same_bank": ("Sender and receiver use the same bank", "flag"), "is_self_transfer": ("Sender and receiver are the same account", "flag"),
    "is_round_amount": ("Amount is a round number", "flag"), "in_amount_band": ("Amount is between 3,000 and 30,000", "flag"),
    "cur_code": ("Payment currency (code)", "code"),
}
for f in cats["pay_format"]:
    labels["fmt_" + f.replace(" ", "_")] = (f"Payment format is {f}", "flag")
sides = {"s_": "Sender", "d_": "Receiver"}
per_side = {
    "out_cnt_24h": ("payments sent in the last 24 hours", "count"), "out_cnt_7d": ("payments sent in the last 7 days", "count"),
    "in_cnt_24h": ("payments received in the last 24 hours", "count"), "in_cnt_7d": ("payments received in the last 7 days", "count"),
    "out_sum_24h": ("total amount sent in the last 24 hours", "amount"), "out_sum_7d": ("total amount sent in the last 7 days", "amount"),
    "in_sum_24h": ("total amount received in the last 24 hours", "amount"), "in_sum_7d": ("total amount received in the last 7 days", "amount"),
    "out_mean_7d": ("average payment sent over the last 7 days", "amount"), "in_mean_7d": ("average payment received over the last 7 days", "amount"),
    "uniq_out": ("different accounts paid so far", "count"), "uniq_in": ("different accounts received from so far", "count"),
    "tot_out": ("payments sent so far in the data", "count"), "tot_in": ("payments received so far in the data", "count"),
    "gap_s": ("time since the account's previous activity", "seconds"),
}
for p, who in sides.items():
    for k, (txt, unit) in per_side.items():
        labels[p + k] = (f"{who}: {txt}", unit)
    labels[p + "pagerank"] = (f"{who}: importance in the payment network (PageRank, 1 = average)", "score")
labels["has_reverse_before"] = ("The receiver has already paid the sender before", "flag")
missing = [f for f in feature_names if f not in labels]
assert not missing, f"no plain-language label for: {missing}"
feature_labels = {f: {"label": labels[f][0], "unit": labels[f][1]} for f in feature_names}

categories = {"pay_format": cats["pay_format"], "currencies": {c: i for i, c in enumerate(cats["cur_paid"])}, "amount_band": [3000, 30000],
              "round_amount_rule": "amount divisible by 100", "feature_names": feature_names, "tabular_features": tab_cols, "history_features": FEATURES_HISTORY}

tr = train.filter(pl.col("ts") >= pl.datetime(2022, 9, 2))                       # same rows the models were fitted on after burn-in-free training
S_COLS = [c for c in FEATURES_HISTORY if c.startswith("s_")] + ["has_reverse_before"]
D_COLS = [c for c in FEATURES_HISTORY if c.startswith("d_")]
has_hist = tr.filter((pl.col("s_tot_out").cast(pl.Int64) + pl.col("s_tot_in").cast(pl.Int64)) > 0)
q90 = float(has_hist["s_out_cnt_24h"].quantile(0.90))
groups = {
    "new_account": ("A brand-new account with no earlier activity", (pl.col("s_tot_out").cast(pl.Int64) + pl.col("s_tot_in").cast(pl.Int64)) == 0),
    "typical_customer": ("A typical customer: at most one payment in the last day, a few regular recipients",
                         (pl.col("s_out_cnt_24h") <= 1) & (pl.col("s_tot_out") >= 1) & (pl.col("s_uniq_out") <= 10)),
    "busy_business": (f"A busy business: at least {q90:.0f} payments in the last 24 hours (top 10%)", pl.col("s_out_cnt_24h") >= q90),
    "rapid_sender": ("A rapid sender: 5 or more payments in the last day, the latest within 2 minutes", (pl.col("s_out_cnt_24h") >= 5) & (pl.col("s_gap_s") <= 120)),
    "dormant_returning": ("A dormant account returning after 2+ days of silence", (pl.col("s_gap_s") >= 2 * 86400) & (pl.col("s_tot_out") >= 1)),
}
profiles = {}
for name, (desc, cond) in groups.items():
    g = tr.filter(cond)
    if g.height < 100:
        print(f"profile {name}: only {g.height} rows, skipped")
        continue
    med = g.select([pl.col(c).median().alias(c) for c in S_COLS]).to_dicts()[0]
    profiles[name] = {"description": desc, "n_rows": g.height, "laundering_rate": float(g["label"].mean()),
                      "features": {k: (None if v is None else float(v)) for k, v in med.items()}}
recv = tr.filter((pl.col("d_tot_out").cast(pl.Int64) + pl.col("d_tot_in").cast(pl.Int64)) > 0)
receiver_default = {k: (None if v is None else float(v)) for k, v in recv.select([pl.col(c).median().alias(c) for c in D_COLS]).to_dicts()[0].items()}
sliders = {k: {"min": 0.0, "max": float(has_hist[k].quantile(0.99)), "default": float(has_hist[k].median())} for k in ("s_out_cnt_24h", "s_uniq_out")}
profiles_json = {"profiles": profiles, "receiver_default": receiver_default, "sliders": sliders,
                 "note": "Illustrative scenarios: medians of training rows that match each description. They are not real accounts."}
print({k: (v["n_rows"], round(v["laundering_rate"], 4)) for k, v in profiles.items()})


In [ ]:
import time as _t; print("[cell 9: write] started", _t.strftime("%H:%M:%S"))
ids = pl.concat([edges.select("txn_id"), history.select("txn_id"), top_txn.select("txn_id")]).unique()
txns = E.join(ids, on="txn_id", how="semi").with_columns(pl.col("typology").alias("typology_truth")).drop("typology")
edges.write_parquet(f"{OUT}/edges.parquet")
history.write_parquet(f"{OUT}/history.parquet")
txns.write_parquet(f"{OUT}/txns.parquet")
shap.write_parquet(f"{OUT}/shap.parquet")

alerts = (acct.join(showcase, left_on="acct", right_on="account").rename({"acct": "account"})
              .with_columns(pl.col("typologies").list.join("; ").alias("typologies_truth"))
              .drop("typologies"))
for m in ("blend", "xgb", "gnn"):
    alerts = alerts.with_columns(pl.col(f"{m}_risk").rank("ordinal", descending=True).cast(pl.Int32).alias(f"{m}_rank"))


def human(value, unit):
    """A value in words a non-technical reader understands."""
    if value is None:
        return "not available"
    if unit == "seconds":
        return f"{value / 3600:.1f} hours" if value >= 3600 else f"{value / 60:.0f} minutes"
    if unit == "flag":
        return "yes" if value >= 0.5 else "no"
    return f"{value:,.4g}"


def why(account):
    parts = []
    for row in shap.filter(pl.col("account") == account).head(2).to_dicts():
        info = feature_labels[row["feature"]]
        parts.append(f"{info['label']}: {human(row['value'], info['unit'])} ({'raises' if row['contribution'] > 0 else 'lowers'} risk)")
    return "; ".join(parts)


alerts = alerts.with_columns(pl.col("account").map_elements(why, return_dtype=pl.String).alias("why")).sort("blend_rank")
alerts.write_parquet(f"{OUT}/alerts.parquet")

# truth for the agent evaluation (hidden in the UI)
truth = (T.filter(pl.col("label") == 1).select(account=pl.col("src"), typology=pl.col("typology")).unique()
          .join(showcase.select("account"), on="account", how="semi"))
truth.write_parquet(f"{OUT}/truth.parquet")

for f in xs["seeds"]:
    shutil.copy(find(f"xgb_v2_final_seed{f}.json"), f"{OUT}/model/xgb_seed{f}.json")
thr_p = float(1 / (1 + np.exp(-(THR["xgb"] * R["xgb_sd"] + R["xgb_mu"]))))                       # z-scale threshold -> probability
json.dump({"feature_names": feature_names, "threshold": thr_p, "model": "xgb_v2_final_ensemble3",
           "files": [f"xgb_seed{s}.json" for s in xs["seeds"]]}, open(f"{OUT}/model/meta.json", "w"), indent=1)
json.dump({"recipe": R, "thresholds_z": THR, "note": "z = (logit(p) - mu) / sd with the validation constants; blend = w_gnn * gnn_z + (1 - w_gnn) * xgb_z"},
          open(f"{OUT}/blend.json", "w"), indent=1)
json.dump(feature_labels, open(f"{OUT}/feature_labels.json", "w"), indent=1)
json.dump(profiles_json, open(f"{OUT}/profiles.json", "w"), indent=1)
json.dump(categories, open(f"{OUT}/categories.json", "w"), indent=1)

# metrics for the model card: test numbers recomputed from the bundle's scores, PR curves, recall per typology
yt, curves, model_rows = test_sc["label"].to_numpy(), {}, {}
for name, col in (("xgb", "xgb_z"), ("gnn", "gnn_z"), ("blend", "blend_z")):
    s_ = test_sc[col].to_numpy()
    e = evaluate(yt, s_, THR[name])
    model_rows[name] = {k: float(e[k]) for k in ("pr_auc", "f1", "recall_at_1pct_fpr", "precision_at_1000")}
    p_, r_, _ = precision_recall_curve(yt, s_)
    idx = np.unique(np.linspace(0, len(p_) - 1, 300).astype(int))
    curves[name] = {"precision": p_[idx].tolist(), "recall": r_[idx].tolist()}
static = {"rules_v1": {"pr_auc": 0.008, "f1": 0.030, "recall_at_1pct_fpr": 0.012, "precision_at_1000": 0.0},
          "xgb_tabular": {"pr_auc": 0.065, "f1": 0.148, "recall_at_1pct_fpr": 0.454, "precision_at_1000": 0.145}}
json.dump({"test_window": "2022-09-09 to 2022-09-10", "n_test_transactions": int(len(yt)), "n_test_positives": int(yt.sum()),
           "models": {**static, **model_rows}, "pr_curves": curves, "typology_recall": bs["typology_recall"],
           "overlap_at_1pct_fpr": bs["overlap_at_1pct_fpr"], "thresholds_z": THR,
           "definitions": {"xgb": "average probability of 3 XGBoost models", "gnn": "average probability of 3 GNN seeds",
                           "blend": "0.4 GNN + 0.6 XGBoost on standardised logits (weight chosen on validation)"}},
          open(f"{OUT}/metrics.json", "w"), indent=1)

total = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(OUT) for f in fs)
for dp, _, fs in os.walk(OUT):
    for f in sorted(fs):
        print(f"{os.path.relpath(os.path.join(dp, f), OUT):28s} {os.path.getsize(os.path.join(dp, f)) / 1e6:7.2f} MB")
print(f"total {total / 1e6:.1f} MB")
assert total < 50e6, "the demo bundle must stay under 50 MB"
shutil.make_archive("/kaggle/working/demo_bundle", "zip", OUT)


In [ ]:
import time as _t; print("[cell 10: log] started", _t.strftime("%H:%M:%S"))
with mlflow.start_run(run_name="demo_bundle"):
    mlflow.log_params({"showcase_accounts": len(chosen), "max_txn_per_account": MAX_TXN, "edge_cap": EDGE_CAP, "bundle_mb": round(total / 1e6, 1)})
    mlflow.log_metrics({f"{n}_{k}": v for n, r in model_rows.items() for k, v in r.items()})
    mlflow.log_artifact("/kaggle/working/demo_bundle.zip")
    mlflow.log_artifact(f"{OUT}/blend.json")
    mlflow.log_artifact(f"{OUT}/metrics.json")
    try:
        for f in sorted(os.listdir(f"{OUT}/model")):
            mlflow.log_artifact(f"{OUT}/model/{f}", "model")
        uri = f"runs:/{mlflow.active_run().info.run_id}/model"
        mv = mlflow.register_model(uri, "papertrail-scorer")
        print("registered model papertrail-scorer, version", mv.version)
    except Exception as e:  # noqa: BLE001
        print("model registry step skipped (artifacts are still logged):", repr(e)[:200])
print(alerts.select("account", "tag", "blend_rank", "blend_risk", "n_flag_blend", "n_laundering").head(8))
print("Download demo_bundle.zip from the Output tab and unzip it into PaperTrail/data/demo/")
